# Notebook 10: Autism Brain Transcriptomics — Real-Data Subtyping

**Dataset:** [GSE28521](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521) — Voineagu et al. 2011, *Nature*  
**Title:** "Transcriptomic Analysis of Autism Brain Reveals Convergent Molecular Pathology"  
**Platform:** Illumina HumanRef-8 v3.0 (GPL6883)  
**Samples:** 79 post-mortem brain samples (ASD + controls) across 3 brain regions  
- Frontal cortex (BA9)  
- Temporal cortex (BA41/42)  
- Cerebellum  

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0  
**Author:** Rohit Chauhan ([ORCID: 0009-0003-9895-4629](https://orcid.org/0009-0003-9895-4629))  
**Zenodo DOI:** [10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)  

---

## What this notebook does

1. Downloads GSE28521 expression data from GEO
2. Preprocesses the expression matrix (probe → gene symbol mapping, QC)
3. Runs pathway-level scoring using 15 curated autism pathways (SFARI-derived)
4. Discovers molecular subtypes via GMM clustering
5. Validates subtypes through 3 validation gates (label shuffle, random gene sets, bootstrap stability)
6. Characterizes subtypes (enriched pathways, top contributing genes)
7. Benchmarks against NMF, PCA+K-means, gene-level K-means, and random baseline
8. Generates publication-quality figures

**Runtime:** ~5-10 minutes on Colab Pro

## 1. Setup & Installation

In [ ]:
# Install pathway-subtyping framework with visualization extras
!pip install -q pathway-subtyping[viz]==0.3.0 GEOparse

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Framework imports
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    ClusteringAlgorithm,
    select_n_clusters,
    compare_algorithms,
    ValidationGates,
    characterize_subtypes,
    generate_subtype_heatmap,
    generate_gene_heatmap,
    export_characterization,
    run_benchmark_comparison,
    plot_static_scatter,
    DimReductionMethod,
    compute_dim_reduction,
)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directory
OUTPUT_DIR = "./outputs/gse28521"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

## 2. Download GSE28521 from GEO

We use the `GEOparse` library to download the series matrix file, which contains
probe-level expression values and sample metadata.

In [ ]:
import GEOparse

# Download the dataset (cached after first run)
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading GSE28521 from GEO (this may take a minute)...")
gse = GEOparse.get_GEO(geo="GSE28521", destdir=DATA_DIR, silent=True)
print(f"Downloaded. Platform(s): {list(gse.gpls.keys())}")
print(f"Number of samples: {len(gse.gsms)}")

## 3. Extract Sample Metadata

Parse sample names to extract diagnosis (ASD vs Control) and brain region.

In [ ]:
# Extract metadata from sample characteristics
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    char_dict = {}
    for c in chars:
        if ":" in c:
            key, val = c.split(":", 1)
            char_dict[key.strip()] = val.strip()
    
    # Parse sample title for diagnosis and region
    title = gsm.metadata.get("title", [""])[0]
    
    metadata_rows.append({
        "sample_id": gsm_name,
        "title": title,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index("sample_id")

# Determine diagnosis and region from title or characteristics
# Voineagu naming: A_DONORNAME_REGION (Autism) or C_DONORNAME_REGION (Control)
# Region codes: C=Cerebellum, F=Frontal, T=Temporal
def parse_sample_info(title):
    """Parse diagnosis and brain region from sample title."""
    parts = title.split("_")
    diagnosis = "ASD" if parts[0] == "A" else "Control" if parts[0] == "C" else "Unknown"
    region_code = parts[-1] if len(parts) >= 3 else "Unknown"
    region_map = {"C": "Cerebellum", "F": "Frontal_Cortex", "T": "Temporal_Cortex"}
    region = region_map.get(region_code, region_code)
    return diagnosis, region

parsed = metadata["title"].apply(parse_sample_info)
metadata["diagnosis"] = [p[0] for p in parsed]
metadata["brain_region"] = [p[1] for p in parsed]

print("\n--- Sample Breakdown ---")
print(metadata.groupby(["brain_region", "diagnosis"]).size().unstack(fill_value=0))
print(f"\nTotal samples: {len(metadata)}")

## 4. Build Expression Matrix

Extract the probe-level expression values from the GEO series matrix,
then map probes to gene symbols using the platform annotation.

In [ ]:
# Extract expression table from the series matrix
# GEOparse provides a pivoted table with probes as rows and samples as columns
expression_df = gse.pivot_samples("VALUE")
print(f"Raw expression matrix: {expression_df.shape[0]} probes x {expression_df.shape[1]} samples")

# Ensure numeric
expression_df = expression_df.apply(pd.to_numeric, errors="coerce")
expression_df = expression_df.dropna(how="all")
print(f"After dropping all-NaN probes: {expression_df.shape[0]} probes")

In [ ]:
# Map probes to gene symbols using platform annotation
gpl = list(gse.gpls.values())[0]
gpl_table = gpl.table

# Find the gene symbol column (varies by platform)
symbol_col = None
for col in ["Symbol", "Gene Symbol", "GENE_SYMBOL", "gene_assignment", "Gene_Symbol"]:
    if col in gpl_table.columns:
        symbol_col = col
        break

if symbol_col is None:
    # For Illumina platforms, gene symbols may be in 'Symbol' column
    print(f"Available columns in platform table: {list(gpl_table.columns)}")
    # Try to use whatever is available
    for col in gpl_table.columns:
        if "symbol" in col.lower() or "gene" in col.lower():
            symbol_col = col
            break

print(f"Using gene symbol column: '{symbol_col}'")

# Create probe-to-gene mapping
probe_to_gene = gpl_table.set_index("ID")[symbol_col].dropna()
probe_to_gene = probe_to_gene[probe_to_gene.str.strip() != ""]
print(f"Probes with gene symbols: {len(probe_to_gene)}")

In [ ]:
# Map probes to genes and collapse (mean of probes per gene)
# Keep only probes that map to known gene symbols
common_probes = expression_df.index.intersection(probe_to_gene.index)
expression_mapped = expression_df.loc[common_probes].copy()
expression_mapped["gene_symbol"] = probe_to_gene.loc[common_probes].values

# Collapse multiple probes per gene by taking the mean
gene_expression = expression_mapped.groupby("gene_symbol").mean()
print(f"Gene-level expression: {gene_expression.shape[0]} genes x {gene_expression.shape[1]} samples")

# Transpose to samples x genes (framework expects this orientation)
gene_expression = gene_expression.T
print(f"Transposed: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")

# Check if already log-transformed (Illumina data is typically already log2)
max_val = gene_expression.max().max()
print(f"Max expression value: {max_val:.2f}")
if max_val > 30:
    print("Data appears to be in raw scale — applying log2(x+1) transform")
    gene_expression = np.log2(gene_expression + 1)
else:
    print("Data appears to be already log-transformed — no additional transform needed")

In [ ]:
# Basic QC
print("--- Expression Matrix QC ---")
print(f"Shape: {gene_expression.shape}")
print(f"Missing values: {gene_expression.isna().sum().sum()}")
print(f"Expression range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]")
print(f"Mean expression: {gene_expression.mean().mean():.2f}")

# Drop genes with zero variance
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f"Dropped {n_zero_var} zero-variance genes. Remaining: {gene_expression.shape[1]}")

# Fill any remaining NaN with column median
if gene_expression.isna().any().any():
    gene_expression = gene_expression.fillna(gene_expression.median())
    print("Filled remaining NaN values with column medians.")

print(f"\nFinal expression matrix: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")

## 5. Load Autism Pathway Gene Sets

The framework ships with 15 curated autism pathway gene sets derived from
SFARI Gene, Satterstrom et al. 2020, and ASC exome studies.

In [ ]:
# Download the GMT file from the framework repo
import urllib.request

GMT_URL = "https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/data/pathways/autism_pathways.gmt"
GMT_PATH = os.path.join(DATA_DIR, "autism_pathways.gmt")

if not os.path.exists(GMT_PATH):
    urllib.request.urlretrieve(GMT_URL, GMT_PATH)
    print(f"Downloaded autism_pathways.gmt")

# Parse GMT file
pathways = {}
with open(GMT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) >= 3:
            pathways[parts[0]] = parts[2:]

print(f"Loaded {len(pathways)} pathways:")
total_genes = set()
for name, genes in pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    total_genes.update(genes)
    print(f"  {name}: {len(genes)} genes ({available} found in expression data)")

print(f"\nTotal unique pathway genes: {len(total_genes)}")
print(f"Found in expression data: {len(total_genes & set(gene_expression.columns))}")

## 6. Pathway Scoring

Score each sample on each pathway using ssGSEA (single-sample Gene Set Enrichment Analysis).
This reduces the ~20,000 gene expression matrix to a 15-pathway score matrix.

In [ ]:
# Score pathways using ssGSEA
scoring_result = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores = scoring_result.pathway_scores

print("\n--- Scoring Report ---")
print(scoring_result.format_report())
print(f"\nPathway score matrix: {pathway_scores.shape}")
print(f"Pathways scored: {scoring_result.n_pathways_scored}")
print(f"Pathways skipped: {scoring_result.n_pathways_skipped}")
if scoring_result.skipped_pathways:
    print(f"Skipped: {scoring_result.skipped_pathways}")

In [ ]:
# Visualize pathway score distribution by diagnosis
scores_with_meta = pathway_scores.copy()
scores_with_meta["diagnosis"] = metadata.loc[pathway_scores.index, "diagnosis"]
scores_with_meta["brain_region"] = metadata.loc[pathway_scores.index, "brain_region"]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for i, pathway in enumerate(pathway_scores.columns):
    ax = axes[i // 5, i % 5]
    for dx, color in [("ASD", "coral"), ("Control", "steelblue")]:
        subset = scores_with_meta[scores_with_meta["diagnosis"] == dx][pathway]
        ax.hist(subset, alpha=0.6, label=dx, color=color, bins=15)
    ax.set_title(pathway.replace("_", "\n"), fontsize=8)
    ax.set_xlabel("")
    if i == 0:
        ax.legend(fontsize=7)
plt.suptitle("Pathway Score Distributions: ASD vs Control", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pathway_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Optimal Cluster Selection

Use BIC (Bayesian Information Criterion) to select the optimal number of molecular subtypes.

In [ ]:
# Select optimal number of clusters using BIC
selection = select_n_clusters(
    data=pathway_scores.values,
    k_range=list(range(2, 8)),
    method="bic",
    seed=SEED,
)

optimal_k = selection.optimal_k
print(f"Optimal k (BIC): {optimal_k}")
print(f"\nBIC values: {selection.bic_values}")
print(f"Silhouette values: {selection.silhouette_values}")

# Plot BIC and Silhouette curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ks = sorted(selection.bic_values.keys())
ax1.plot(ks, [selection.bic_values[k] for k in ks], "bo-", linewidth=2)
ax1.axvline(x=optimal_k, color="red", linestyle="--", label=f"Optimal k={optimal_k}")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("BIC (lower is better)")
ax1.set_title("Model Selection: BIC")
ax1.legend()

ax2.plot(ks, [selection.silhouette_values[k] for k in ks], "go-", linewidth=2)
ax2.axvline(x=optimal_k, color="red", linestyle="--", label=f"Optimal k={optimal_k}")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score (higher is better)")
ax2.set_title("Model Selection: Silhouette")
ax2.legend()

plt.suptitle("GSE28521: Optimal Cluster Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_selection.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. GMM Clustering

Run Gaussian Mixture Model clustering at the optimal k.

In [ ]:
# Run GMM clustering
clustering = run_clustering(
    data=pathway_scores.values,
    n_clusters=optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)

print(f"--- GMM Clustering Results ---")
print(f"k = {clustering.n_clusters}")
print(f"Silhouette score: {clustering.silhouette:.4f}")
print(f"Calinski-Harabasz: {clustering.calinski_harabasz:.2f}")
print(f"Davies-Bouldin: {clustering.davies_bouldin:.4f}")
if clustering.bic is not None:
    print(f"BIC: {clustering.bic:.2f}")
print(f"Converged: {clustering.converged}")

# Show subtype sizes
labels = clustering.labels
print(f"\nSubtype sizes:")
for i in range(optimal_k):
    count = (labels == i).sum()
    print(f"  Subtype {i}: {count} samples ({count/len(labels)*100:.1f}%)")

In [ ]:
# Cross-tabulate subtypes with diagnosis and brain region
subtype_meta = metadata.loc[pathway_scores.index].copy()
subtype_meta["subtype"] = labels

print("--- Subtype × Diagnosis ---")
ct_dx = pd.crosstab(subtype_meta["subtype"], subtype_meta["diagnosis"], margins=True)
print(ct_dx)

print("\n--- Subtype × Brain Region ---")
ct_region = pd.crosstab(subtype_meta["subtype"], subtype_meta["brain_region"], margins=True)
print(ct_region)

print("\n--- Subtype × Diagnosis × Region ---")
ct_full = pd.crosstab([subtype_meta["subtype"], subtype_meta["brain_region"]], subtype_meta["diagnosis"])
print(ct_full)

In [ ]:
# PCA scatter plot colored by subtype, with diagnosis markers
embedding, pca_meta = compute_dim_reduction(
    pathway_scores=pathway_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Color by subtype
scatter_colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                    label=f"Subtype {i} (n={mask.sum()})", s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[0].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[0].set_title("Colored by Molecular Subtype")
axes[0].legend()

# Plot 2: Color by diagnosis
dx_colors = {"ASD": "coral", "Control": "steelblue"}
for dx, color in dx_colors.items():
    mask = subtype_meta["diagnosis"].values == dx
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=color,
                    label=dx, s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[1].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[1].set_title("Colored by Diagnosis")
axes[1].legend()

# Plot 3: Color by brain region
region_colors = {"Frontal_Cortex": "#2ecc71", "Temporal_Cortex": "#e74c3c", "Cerebellum": "#3498db"}
for region, color in region_colors.items():
    mask = subtype_meta["brain_region"].values == region
    axes[2].scatter(embedding[mask, 0], embedding[mask, 1], c=color,
                    label=region.replace("_", " "), s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[2].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[2].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[2].set_title("Colored by Brain Region")
axes[2].legend()

plt.suptitle(f"GSE28521: Pathway-Based Molecular Subtypes (k={optimal_k}, GMM)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_scatter_trio.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Validation Gates

Run the framework's validation gates to confirm the subtypes are biologically meaningful:
1. **Negative Control 1 (Label Shuffle):** Shuffled labels should NOT be recoverable
2. **Negative Control 2 (Random Gene Sets):** Random pathways should NOT reproduce the clusters
3. **Stability (Bootstrap):** Clusters should survive resampling

In [ ]:
# Run all validation gates
gates = ValidationGates(
    seed=SEED,
    n_permutations=200,
    n_bootstrap=100,
    stability_threshold=0.8,
    null_ari_max=0.15,
    show_progress=True,
)

print("Running validation gates (this may take 1-2 minutes)...")
val_result = gates.run_all(
    pathway_scores=pathway_scores,
    cluster_labels=labels,
    pathways=pathways,
    gene_burdens=gene_expression,
    n_clusters=optimal_k,
    gmm_seed=SEED,
)

print("\n" + "=" * 60)
print("VALIDATION GATES RESULTS")
print("=" * 60)
print(f"\nAll gates passed: {'YES' if val_result.all_passed else 'NO'}")
print()
for gate in val_result.results:
    status = 'PASS' if gate.passed else 'FAIL'
    print(f"  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} "
          f"(threshold: {gate.comparison} {gate.threshold:.4f})")

## 10. Subtype Characterization

Identify which pathways and genes drive each molecular subtype.

In [ ]:
# Characterize subtypes
char_result = characterize_subtypes(
    pathway_scores=pathway_scores,
    cluster_labels=labels,
    gene_burdens=gene_expression,
    pathways=pathways,
    fdr_alpha=0.05,
    top_n_genes=20,
    seed=SEED,
)

# Print characterization report
print(char_result.format_report())

In [ ]:
# Generate heatmaps
fig_heatmap = generate_subtype_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, "subtype_heatmap.png"),
    figsize=(14, 8),
)
plt.show()

fig_genes = generate_gene_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, "gene_heatmap.png"),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

In [ ]:
# Export characterization data to CSV
export_files = export_characterization(
    char_result,
    output_dir=OUTPUT_DIR,
    formats=["csv"],
)
print("Exported files:")
for f in export_files:
    print(f"  {f}")

## 11. Benchmark Comparison

Compare the framework's pathway-based GMM approach against four alternative methods:
- **NMF Clustering**: Non-negative matrix factorization
- **PCA + K-means**: Principal component reduction + K-means
- **Gene-level K-means**: Direct K-means on gene expression (no pathway aggregation)
- **Random Baseline**: Random cluster assignments

In [ ]:
# Run benchmark comparison
print("Running benchmark comparison...")
bench_result = run_benchmark_comparison(
    gene_burdens=gene_expression,
    pathway_scores=pathway_scores,
    pathways=pathways,
    n_clusters=optimal_k,
    seed=SEED,
)

print("\n" + bench_result.format_report())

In [ ]:
# Visualize benchmark results
methods = list(bench_result.method_results.keys())
silhouettes = [bench_result.method_results[m].silhouette for m in methods]
runtimes = [bench_result.method_results[m].runtime_seconds for m in methods]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Silhouette comparison
colors = ["#2ecc71" if m == bench_result.best_method else "#3498db" for m in methods]
bars = ax1.barh(methods, silhouettes, color=colors)
ax1.set_xlabel("Silhouette Score (higher is better)")
ax1.set_title("Clustering Quality: Method Comparison")
for bar, val in zip(bars, silhouettes):
    ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10)

# Runtime comparison
ax2.barh(methods, runtimes, color="#9b59b6")
ax2.set_xlabel("Runtime (seconds)")
ax2.set_title("Computational Cost")
for bar, val in zip(ax2.patches, runtimes):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.2f}s", va="center", fontsize=10)

plt.suptitle(f"GSE28521: Benchmark Comparison (k={optimal_k})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "benchmark_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 12. Region-Specific Analysis

Voineagu et al. found that frontal and temporal cortex showed convergent molecular pathology,
while cerebellum showed distinct patterns. Let's test whether our pathway-based subtypes
capture this biology.

In [ ]:
# Run subtyping within each brain region
region_results = {}
for region in ["Frontal_Cortex", "Temporal_Cortex", "Cerebellum"]:
    region_mask = subtype_meta["brain_region"] == region
    region_scores = pathway_scores.loc[region_mask]
    
    if len(region_scores) < 10:
        print(f"Skipping {region} — too few samples ({len(region_scores)})")
        continue
    
    # Select optimal k for this region
    region_selection = select_n_clusters(
        data=region_scores.values,
        k_range=list(range(2, min(6, len(region_scores) // 3))),
        method="bic",
        seed=SEED,
    )
    
    # Cluster
    region_clustering = run_clustering(
        data=region_scores.values,
        n_clusters=region_selection.optimal_k,
        algorithm=ClusteringAlgorithm.GMM,
        seed=SEED,
    )
    
    region_results[region] = {
        "n_samples": len(region_scores),
        "optimal_k": region_selection.optimal_k,
        "silhouette": region_clustering.silhouette,
        "labels": region_clustering.labels,
        "scores": region_scores,
    }
    
    # Cross-tab with diagnosis
    region_meta = subtype_meta.loc[region_mask]
    ct = pd.crosstab(
        pd.Series(region_clustering.labels, index=region_meta.index, name="subtype"),
        region_meta["diagnosis"],
    )
    print(f"\n{region} (n={len(region_scores)}, k={region_selection.optimal_k}, "
          f"silhouette={region_clustering.silhouette:.3f}):")
    print(ct)

In [ ]:
# Compare pathway score profiles between regions
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, region in enumerate(["Frontal_Cortex", "Temporal_Cortex", "Cerebellum"]):
    region_mask = subtype_meta["brain_region"] == region
    region_data = scores_with_meta[region_mask]
    
    means_asd = region_data[region_data["diagnosis"] == "ASD"][pathway_scores.columns].mean()
    means_ctl = region_data[region_data["diagnosis"] == "Control"][pathway_scores.columns].mean()
    diff = means_asd - means_ctl
    
    colors = ["coral" if v > 0 else "steelblue" for v in diff.values]
    axes[i].barh(range(len(diff)), diff.values, color=colors)
    axes[i].set_yticks(range(len(diff)))
    axes[i].set_yticklabels([p.replace("_", " ") for p in diff.index], fontsize=8)
    axes[i].set_xlabel("ASD - Control (Z-score diff)")
    axes[i].set_title(region.replace("_", " "))
    axes[i].axvline(x=0, color="black", linewidth=0.5)

plt.suptitle("Pathway Score Differences (ASD vs Control) by Brain Region", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "region_pathway_diff.png"), dpi=150, bbox_inches="tight")
plt.show()

## 13. Algorithm Comparison

Compare GMM against K-means, Hierarchical, and Spectral clustering.

In [ ]:
# Compare all clustering algorithms
algo_comparison = compare_algorithms(
    data=pathway_scores.values,
    n_clusters=optimal_k,
    seed=SEED,
)

print(f"Most stable algorithm: {algo_comparison.most_stable_algorithm}")
print(f"\nPairwise ARI (inter-algorithm agreement):")
for pair, ari in algo_comparison.pairwise_ari.items():
    print(f"  {pair}: {ari:.4f}")

print(f"\nPer-algorithm metrics:")
for algo, res in algo_comparison.results.items():
    print(f"  {algo}: silhouette={res.silhouette:.4f}, CH={res.calinski_harabasz:.1f}, DB={res.davies_bouldin:.4f}")

## 14. Summary & Export

Save all results for downstream use in notebooks 13-15.

In [ ]:
# Save key outputs
pathway_scores.to_csv(os.path.join(OUTPUT_DIR, "pathway_scores.csv"))
subtype_meta.to_csv(os.path.join(OUTPUT_DIR, "sample_metadata_with_subtypes.csv"))
gene_expression.to_csv(os.path.join(OUTPUT_DIR, "gene_expression_processed.csv"))

# Save validation and clustering results as JSON
import json

results_summary = {
    "dataset": "GSE28521",
    "citation": "Voineagu et al. 2011, Nature",
    "n_samples": int(len(pathway_scores)),
    "n_genes": int(gene_expression.shape[1]),
    "n_pathways_scored": int(scoring_result.n_pathways_scored),
    "scoring_method": "ssGSEA",
    "optimal_k": int(optimal_k),
    "clustering_algorithm": "GMM",
    "silhouette": float(clustering.silhouette),
    "validation_all_passed": bool(val_result.all_passed),
    "validation_gates": [
        {"name": str(g.name), "passed": bool(g.passed), "metric": str(g.metric_name),
         "value": float(g.metric_value), "threshold": float(g.threshold)}
        for g in val_result.results
    ],
    "benchmark_best_method": str(bench_result.best_method),
    "benchmark_ranking": [str(r) for r in bench_result.ranking],
    "subtype_sizes": {str(i): int((labels == i).sum()) for i in range(optimal_k)},
    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print(f"\nDataset: GSE28521 (Voineagu et al. 2011, Nature)")
print(f"Samples: {len(pathway_scores)} across 3 brain regions")
print(f"Genes: {gene_expression.shape[1]} → {scoring_result.n_pathways_scored} pathways (ssGSEA)")
print(f"Optimal subtypes: {optimal_k} (BIC-selected, GMM)")
print(f"Silhouette: {clustering.silhouette:.4f}")
print(f"Validation: {'ALL PASSED' if val_result.all_passed else 'SOME FAILED'}")
print(f"Best method: {bench_result.best_method}")
print(f"\nOutputs saved to: {OUTPUT_DIR}/")
print(f"  - pathway_scores.csv")
print(f"  - sample_metadata_with_subtypes.csv")
print(f"  - gene_expression_processed.csv")
print(f"  - results_summary.json")
print(f"  - subtype_heatmap.png")
print(f"  - gene_heatmap.png")
print(f"  - pca_scatter_trio.png")
print(f"  - model_selection.png")
print(f"  - benchmark_comparison.png")
print(f"  - region_pathway_diff.png")
print(f"  - pathway_distributions.png")

## 15. Region-Stratified Subtyping: Frontal Cortex Only

**Why this matters:** The full-dataset analysis (k=7) is dominated by brain region effects
(PC1 = 43% variance). The cerebellum shows opposite molecular patterns to cortex, so the
7 subtypes partly reflect anatomy rather than disease biology.

By subsetting to **frontal cortex only** (the brain region most relevant to ASD — executive
function, social cognition, language), we remove the region confound and ask:

> **Do ASD-specific molecular subtypes exist _within_ the same brain region?**

We test k=2, 3, and 4 and run full validation on each.

In [ ]:
# ── 15a. Subset to Frontal Cortex ──────────────────────────────────────────────

FC_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "frontal_cortex")
os.makedirs(FC_OUTPUT_DIR, exist_ok=True)

# Subset expression and pathway scores to frontal cortex only
fc_mask = metadata.loc[pathway_scores.index, "brain_region"] == "Frontal_Cortex"
fc_scores = pathway_scores.loc[fc_mask]
fc_expression = gene_expression.loc[fc_mask]
fc_meta = metadata.loc[fc_scores.index].copy()

print("=" * 60)
print("FRONTAL CORTEX SUBSET")
print("=" * 60)
print(f"\nSamples: {len(fc_scores)}")
print(f"  ASD:     {(fc_meta['diagnosis'] == 'ASD').sum()}")
print(f"  Control: {(fc_meta['diagnosis'] == 'Control').sum()}")
print(f"Genes:    {fc_expression.shape[1]}")
print(f"Pathways: {fc_scores.shape[1]}")

In [ ]:
# ── 15b. Sweep k=2,3,4 — Clustering + Validation + Characterization ───────────

fc_all_results = {}

for k in [2, 3, 4]:
    print(f"\n{'='*60}")
    print(f"FRONTAL CORTEX — k={k}")
    print(f"{'='*60}")
    
    # Cluster
    fc_clustering = run_clustering(
        data=fc_scores.values,
        n_clusters=k,
        algorithm=ClusteringAlgorithm.GMM,
        seed=SEED,
    )
    fc_labels = fc_clustering.labels
    
    print(f"\nSilhouette: {fc_clustering.silhouette:.4f}")
    print(f"Calinski-Harabasz: {fc_clustering.calinski_harabasz:.2f}")
    print(f"Davies-Bouldin: {fc_clustering.davies_bouldin:.4f}")
    
    # Subtype sizes × diagnosis
    fc_subtype_meta = fc_meta.copy()
    fc_subtype_meta["subtype"] = fc_labels
    ct = pd.crosstab(fc_subtype_meta["subtype"], fc_subtype_meta["diagnosis"], margins=True)
    print(f"\nSubtype × Diagnosis:")
    print(ct)
    
    # Validation gates
    fc_gates = ValidationGates(
        seed=SEED,
        n_permutations=200,
        n_bootstrap=100,
        stability_threshold=0.8,
        null_ari_max=0.15,
        show_progress=False,
    )
    
    fc_val = fc_gates.run_all(
        pathway_scores=fc_scores,
        cluster_labels=fc_labels,
        pathways=pathways,
        gene_burdens=fc_expression,
        n_clusters=k,
        gmm_seed=SEED,
    )
    
    print(f"\nValidation Gates:")
    n_passed = 0
    for gate in fc_val.results:
        status = "PASS" if gate.passed else "FAIL"
        if gate.passed:
            n_passed += 1
        print(f"  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} "
              f"(threshold: {gate.comparison} {gate.threshold:.4f})")
    print(f"\nGates passed: {n_passed}/{len(fc_val.results)} — All passed: {fc_val.all_passed}")
    
    # Characterize
    fc_char = characterize_subtypes(
        pathway_scores=fc_scores,
        cluster_labels=fc_labels,
        gene_burdens=fc_expression,
        pathways=pathways,
        fdr_alpha=0.05,
        top_n_genes=15,
        seed=SEED,
    )
    
    # Store results
    fc_all_results[k] = {
        "clustering": fc_clustering,
        "labels": fc_labels,
        "validation": fc_val,
        "characterization": fc_char,
        "n_gates_passed": n_passed,
        "silhouette": fc_clustering.silhouette,
        "meta": fc_subtype_meta,
    }

In [ ]:
# ── 15c. Compare k=2,3,4 — Pick the best ─────────────────────────────────────

print("=" * 60)
print("FRONTAL CORTEX: k COMPARISON SUMMARY")
print("=" * 60)
print(f"\n{'k':<4} {'Silhouette':<12} {'Gates Passed':<14} {'All Passed':<12}")
print("-" * 42)
for k in [2, 3, 4]:
    r = fc_all_results[k]
    print(f"{k:<4} {r['silhouette']:<12.4f} {r['n_gates_passed']}/{len(r['validation'].results):<12} "
          f"{'YES' if r['validation'].all_passed else 'NO'}")

# Select best k: prioritize validation gates passed, then silhouette
best_fc_k = max(fc_all_results.keys(),
                key=lambda k: (fc_all_results[k]["n_gates_passed"],
                               fc_all_results[k]["silhouette"]))

print(f"\nBest k for frontal cortex: {best_fc_k}")
print(f"  Silhouette: {fc_all_results[best_fc_k]['silhouette']:.4f}")
print(f"  Gates passed: {fc_all_results[best_fc_k]['n_gates_passed']}")

# Compare to full-dataset analysis
print(f"\n--- Comparison: Full Dataset vs Frontal Cortex ---")
print(f"  Full dataset:    k={optimal_k}, silhouette={clustering.silhouette:.4f}, "
      f"gates={sum(1 for g in val_result.results if g.passed)}/{len(val_result.results)}")
print(f"  Frontal cortex:  k={best_fc_k}, silhouette={fc_all_results[best_fc_k]['silhouette']:.4f}, "
      f"gates={fc_all_results[best_fc_k]['n_gates_passed']}/{len(fc_all_results[best_fc_k]['validation'].results)}")

In [ ]:
# ── 15d. Visualize best frontal cortex subtypes ──────────────────────────────

best_r = fc_all_results[best_fc_k]
best_labels = best_r["labels"]
best_char = best_r["characterization"]
best_meta = best_r["meta"]

# PCA scatter: subtypes vs diagnosis (frontal cortex only)
fc_embedding, fc_pca_meta = compute_dim_reduction(
    pathway_scores=fc_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: color by subtype
colors_k = plt.cm.Set1(np.linspace(0, 1, best_fc_k))
for i in range(best_fc_k):
    mask = best_labels == i
    n_asd = (best_meta.loc[fc_scores.index[mask], "diagnosis"] == "ASD").sum()
    n_ctl = mask.sum() - n_asd
    axes[0].scatter(fc_embedding[mask, 0], fc_embedding[mask, 1], c=[colors_k[i]],
                    label=f"FC-Subtype {i} (n={mask.sum()}: {n_asd}A/{n_ctl}C)",
                    s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
axes[0].set_xlabel(f"PC1 ({fc_pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({fc_pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[0].set_title(f"Frontal Cortex Subtypes (k={best_fc_k})")
axes[0].legend(fontsize=9)

# Right: color by diagnosis
for dx, color in [("ASD", "coral"), ("Control", "steelblue")]:
    mask = fc_meta.loc[fc_scores.index, "diagnosis"].values == dx
    axes[1].scatter(fc_embedding[mask, 0], fc_embedding[mask, 1], c=color,
                    label=dx, s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
axes[1].set_xlabel(f"PC1 ({fc_pca_meta['explained_variance_ratio'][0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({fc_pca_meta['explained_variance_ratio'][1]*100:.1f}%)")
axes[1].set_title("Frontal Cortex — Diagnosis")
axes[1].legend(fontsize=9)

plt.suptitle(f"Frontal Cortex Only (n={len(fc_scores)}): Region Confound Removed",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FC_OUTPUT_DIR, "fc_pca_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

# Subtype pathway heatmap
fig_fc_heatmap = generate_subtype_heatmap(
    best_char,
    output_path=os.path.join(FC_OUTPUT_DIR, "fc_subtype_heatmap.png"),
    figsize=(14, max(4, best_fc_k * 1.5)),
)
plt.show()

# Gene contribution heatmap
fig_fc_genes = generate_gene_heatmap(
    best_char,
    output_path=os.path.join(FC_OUTPUT_DIR, "fc_gene_heatmap.png"),
    figsize=(16, max(5, best_fc_k * 2)),
    top_n=15,
)
plt.show()

In [ ]:
# ── 15e. Characterization report for best k ───────────────────────────────────

print("=" * 60)
print(f"FRONTAL CORTEX SUBTYPE CHARACTERIZATION (k={best_fc_k})")
print("=" * 60)
print(best_char.format_report())

# Export characterization CSVs
fc_export = export_characterization(
    best_char,
    output_dir=FC_OUTPUT_DIR,
    formats=["csv"],
)
print("\nExported frontal cortex characterization:")
for f in fc_export:
    print(f"  {f}")

In [ ]:
# ── 15f. Benchmark on frontal cortex only ─────────────────────────────────────

print("Running benchmark comparison (frontal cortex only)...")
fc_bench = run_benchmark_comparison(
    gene_burdens=fc_expression,
    pathway_scores=fc_scores,
    pathways=pathways,
    n_clusters=best_fc_k,
    seed=SEED,
)
print("\n" + fc_bench.format_report())

# Visualize
fc_methods = list(fc_bench.method_results.keys())
fc_sils = [fc_bench.method_results[m].silhouette for m in fc_methods]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71" if m == fc_bench.best_method else "#3498db" for m in fc_methods]
bars = ax.barh(fc_methods, fc_sils, color=colors)
ax.set_xlabel("Silhouette Score (higher is better)")
ax.set_title(f"Frontal Cortex Benchmark (k={best_fc_k})")
for bar, val in zip(bars, fc_sils):
    ax.text(max(bar.get_width() + 0.005, 0.01), bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FC_OUTPUT_DIR, "fc_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 15g. Save frontal cortex results ──────────────────────────────────────────

# Save metadata with frontal cortex subtypes
fc_save_meta = best_meta.copy()
fc_save_meta.to_csv(os.path.join(FC_OUTPUT_DIR, "fc_sample_metadata_with_subtypes.csv"))
fc_scores.to_csv(os.path.join(FC_OUTPUT_DIR, "fc_pathway_scores.csv"))

# Save summary JSON
fc_summary = {
    "analysis": "frontal_cortex_only",
    "dataset": "GSE28521",
    "brain_region": "Frontal_Cortex (BA9)",
    "n_samples": int(len(fc_scores)),
    "n_asd": int((fc_meta["diagnosis"] == "ASD").sum()),
    "n_control": int((fc_meta["diagnosis"] == "Control").sum()),
    "k_tested": [2, 3, 4],
    "best_k": int(best_fc_k),
    "results_by_k": {
        str(k): {
            "silhouette": float(fc_all_results[k]["silhouette"]),
            "n_gates_passed": int(fc_all_results[k]["n_gates_passed"]),
            "all_gates_passed": bool(fc_all_results[k]["validation"].all_passed),
            "validation_gates": [
                {"name": str(g.name), "passed": bool(g.passed),
                 "metric": str(g.metric_name), "value": float(g.metric_value),
                 "threshold": float(g.threshold)}
                for g in fc_all_results[k]["validation"].results
            ],
        }
        for k in [2, 3, 4]
    },
    "best_k_benchmark_winner": str(fc_bench.best_method),
    "comparison_to_full_dataset": {
        "full_k": int(optimal_k),
        "full_silhouette": float(clustering.silhouette),
        "full_gates_passed": int(sum(1 for g in val_result.results if g.passed)),
        "fc_k": int(best_fc_k),
        "fc_silhouette": float(fc_all_results[best_fc_k]["silhouette"]),
        "fc_gates_passed": int(fc_all_results[best_fc_k]["n_gates_passed"]),
    },
    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(FC_OUTPUT_DIR, "fc_results_summary.json"), "w") as f:
    json.dump(fc_summary, f, indent=2)

print("\n" + "=" * 60)
print("FRONTAL CORTEX ANALYSIS COMPLETE")
print("=" * 60)
print(f"\nRegion: Frontal Cortex (BA9)")
print(f"Samples: {len(fc_scores)} ({(fc_meta['diagnosis'] == 'ASD').sum()} ASD, "
      f"{(fc_meta['diagnosis'] == 'Control').sum()} Control)")
print(f"Best k: {best_fc_k}")
print(f"Silhouette: {fc_all_results[best_fc_k]['silhouette']:.4f}")
print(f"Validation gates passed: {fc_all_results[best_fc_k]['n_gates_passed']}")
print(f"Benchmark winner: {fc_bench.best_method}")
print(f"\nOutputs saved to: {FC_OUTPUT_DIR}/")
print(f"  - fc_results_summary.json")
print(f"  - fc_sample_metadata_with_subtypes.csv")
print(f"  - fc_pathway_scores.csv")
print(f"  - fc_subtype_heatmap.png")
print(f"  - fc_gene_heatmap.png")
print(f"  - fc_pca_scatter.png")
print(f"  - fc_benchmark.png")
print(f"  - subtype_summary.csv, pathway_enrichment.csv, gene_contributions.csv")

---

## References

1. Voineagu I, et al. (2011). Transcriptomic analysis of autistic brain reveals convergent molecular pathology. *Nature*, 474(7351):380-384. [PMID: 21614001](https://pubmed.ncbi.nlm.nih.gov/21614001/)
2. Satterstrom FK, et al. (2020). Large-Scale Exome Sequencing Study Implicates Both Developmental and Functional Changes in the Neurobiology of Autism. *Cell*, 180(3):568-584. [PMID: 31981491](https://pubmed.ncbi.nlm.nih.gov/31981491/)
3. Chauhan R (2026). Pathway Subtyping Framework v0.3.0. *Zenodo*. [DOI: 10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)

## Data Availability

- **GSE28521:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521
- **Framework:** https://github.com/topmist-admin/pathway-subtyping-framework
- **PyPI:** `pip install pathway-subtyping`

## License

This notebook is released under CC-BY 4.0.